# Women's E-Commerce Clothing Reviews Analysis

**Team:** Tirth Trivedi (IU2441230646) and Dhairya Shah (IU2441230652)

Run cells from top to bottom. Put the CSV in the `dataset` folder.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path
file_path = '../dataset/Womens Clothing E-Commerce Reviews.csv'
df = pd.read_csv(file_path)
print('Original shape:', df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False))
print('Duplicates with exported index:', df.duplicated().sum())
display(df.dtypes)

## Cleaning
Missing review text is left blank, missing categories are marked Unknown. Numeric values are validated. IQR outliers are flagged but not automatically deleted.

In [ ]:
data=df.drop(columns=['Unnamed: 0'],errors='ignore').copy()
for c in ['Title','Review Text']: data[c]=data[c].fillna('').astype(str)
for c in ['Division Name','Department Name','Class Name']: data[c]=data[c].fillna('Unknown').astype(str).str.strip()
numeric=['Clothing ID','Age','Rating','Recommended IND','Positive Feedback Count']
for c in numeric: data[c]=pd.to_numeric(data[c],errors='coerce')
data.loc[~data['Age'].between(13,100),'Age']=np.nan
data.loc[~data['Rating'].between(1,5),'Rating']=np.nan
data.loc[~data['Recommended IND'].isin([0,1]),'Recommended IND']=np.nan
data.loc[data['Positive Feedback Count']<0,'Positive Feedback Count']=np.nan
data=data.dropna(subset=numeric).drop_duplicates()
for c in numeric: data[c]=data[c].astype(int)
print('Cleaned shape:',data.shape)
print('Missing values remaining:',int(data.isna().sum().sum()))
print('Duplicate rows remaining:',int(data.duplicated().sum()))

In [ ]:
def iqr_bounds(s):
    q1,q3=s.quantile([.25,.75]); iqr=q3-q1
    return q1-1.5*iqr,q3+1.5*iqr
for c in ['Age','Positive Feedback Count']:
    low,high=iqr_bounds(data[c])
    print(c,'IQR bounds:',low,high,'flagged:',int(((data[c]<low)|(data[c]>high)).sum()))
print('IQR flags are reviewed, not automatically removed.')

## Eight visualizations

In [ ]:
# 1. Rating distribution
data['Rating'].value_counts().sort_index().plot(kind='bar',figsize=(8,5)); plt.title('Customer Rating Distribution'); plt.xlabel('Rating (1–5)'); plt.ylabel('Reviews'); plt.xticks(rotation=0); plt.show()

# 2. Recommended vs not recommended
data['Recommended IND'].map({1:'Recommended',0:'Not recommended'}).value_counts().plot(kind='bar',figsize=(7,5)); plt.title('Recommended vs Not Recommended'); plt.ylabel('Reviews'); plt.xticks(rotation=0); plt.show()

# 3. Average rating by department
data.groupby('Department Name')['Rating'].mean().sort_values().plot(kind='barh',figsize=(9,5)); plt.title('Average Rating by Department'); plt.xlabel('Average rating'); plt.xlim(0,5); plt.show()

# 4. Top-rated clothing classes with at least 30 reviews
cs=data.groupby('Class Name')['Rating'].agg(['mean','count']); cs=cs[cs['count']>=30].sort_values('mean',ascending=False).head(10).sort_values('mean'); cs['mean'].plot(kind='barh',figsize=(10,6)); plt.title('Top-Rated Clothing Classes (30+ reviews)'); plt.xlabel('Average rating'); plt.xlim(0,5); plt.show()

# 5. Customer age distribution
data['Age'].plot(kind='hist',bins=20,figsize=(8,5),edgecolor='black'); plt.title('Customer Age Distribution'); plt.xlabel('Age'); plt.ylabel('Reviews'); plt.show()

# 6. Positive feedback count; cap at 30 for display only
data['Positive Feedback Count'].clip(upper=30).plot(kind='hist',bins=16,figsize=(8,5),edgecolor='black'); plt.title('Positive Feedback Counts (30 means 30+)'); plt.xlabel('Feedback count'); plt.ylabel('Reviews'); plt.show()

# 7. Review length by rating
data=data.copy(); data['Review_Length']=data['Review Text'].str.len(); data.boxplot(column='Review_Length',by='Rating',figsize=(8,5),grid=False); plt.suptitle(''); plt.title('Review Length by Rating'); plt.xlabel('Rating'); plt.ylabel('Characters'); plt.show()

# 8. Common review words (not sentiment analysis)
tokens=re.findall(r'\\b[a-z]{3,}\\b',' '.join(data['Review Text'].astype(str)).lower())
stop=set('the and for that this with was but are you have had not she her his they them their from were will would can all about very just really too its one out get got fit size dress love great good nice wear wore look like more when what which who how did because been into also than then there here still only some much little'.split())
freq=pd.Series([w for w in tokens if w not in stop]).value_counts().head(15).sort_values(); freq.plot(kind='barh',figsize=(9,6)); plt.title('Most Frequent Review Words'); plt.xlabel('Count'); plt.show()

## Summary findings

In [ ]:
print('Cleaned records:',len(data))
print(f'Mean rating: {data.Rating.mean():.2f}/5')
print(f'Ratings 4 or 5: {(data.Rating>=4).mean()*100:.1f}%')
print('Recommendation rate:',round((data['Recommended IND'].eq(1)).mean()*100,1),'%')
print(f'Mean age: {data.Age.mean():.1f}; median age: {data.Age.median():.0f}')
display(data.groupby('Department Name').Rating.agg(['mean','count']).sort_values('mean',ascending=False))
print('Age-rating correlation:',round(data.Age.corr(data.Rating),3))
data.drop(columns=['Review_Length'],errors='ignore').to_csv('../dataset/cleaned_womens_clothing_reviews.csv',index=False)